# 🦁 MUFASA — ADTC Profiler Benchmark (Multi-Model)
**Africa Deep Tech Challenge 2026**

| Model | HF Repo / Artifact | Quant | Backend |
|---|---|---|---|
| Qwen3-4B Thinking | Qwen/Qwen3-4B-GGUF | Q4_K_M | llama-cpp-python |
| Phi-4 Mini Reasoning | lmstudio-community/Phi-4-mini-reasoning-GGUF | Q4_K_M | llama-cpp-python |
| Gemma-3-4B-IT | lmstudio-community/gemma-3-4b-it-GGUF | Q4_K_M | llama-cpp-python |
| **Nanbeige4.2-3B** | Nanbeige/Nanbeige4.2-3B via Abiray GGUF quant | **Q4_K_M** | Nanbeige llama.cpp CLI |
| Bonsai-27B | prism-ml/Bonsai-27B-gguf | Q1_0 (ternary) | PrismML llama.cpp CLI |

**Constraint:** 8 GB RAM · CPU-only  
**Metrics:** throughput (tok/s) · TTFT · peak total notebook RSS · elapsed time  
**Profiler:** [adtc-profiler](https://github.com/Africa-Deep-Tech-Foundation/adtc-profiler)

> ⚠️ Enable **Internet** in Kaggle notebook settings before running.

### Strategy
- Standard GGUF models use stock `llama-cpp-python`, CPU-only.
- Nanbeige uses the `nanbeige42` llama.cpp branch because its looped-transformer architecture may not be supported by the stock Python wheel.
- The official BF16 Nanbeige checkpoint is not loaded under the 8 GB limit; a Q4_K_M GGUF quantization is used instead.
- Bonsai-27B Q1_0 uses the PrismML llama.cpp fork.
- Models are downloaded, benchmarked, and released one at a time.
- CLI memory measurement includes the spawned model process, fixing the under-reporting in the original notebook.


In [ ]:
import subprocess, sys, os, shutil
from pathlib import Path

# -----------------------------
# Phase 1: Python dependencies
# -----------------------------
print("📦 Installing packages...")
pkgs = [
    "huggingface_hub>=0.30.0", "psutil", "pandas", "matplotlib",
    "tabulate", "llama-cpp-python",
    "git+https://github.com/Africa-Deep-Tech-Foundation/adtc-profiler.git",
]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--prefer-binary"
] + pkgs)
print("✅ Python packages installed")

# -----------------------------
# Phase 2: CPU llama.cpp builds
# -----------------------------
WORK_DIR = Path("/kaggle/working")
MAX_BUILD_JOBS = max(1, min(8, os.cpu_count() or 4))


def find_binary(build_dir, name):
    candidates = [build_dir / "bin" / name, build_dir / name]
    for path in candidates:
        if path.is_file():
            os.chmod(path, 0o755)
            return str(path)
    for path in build_dir.rglob(name):
        if path.is_file():
            os.chmod(path, 0o755)
            return str(path)
    return None


def build_llama_cpp(label, repo_url, repo_dir, branch=None):
    """Clone and build only the CPU CLI/bench targets. Failure is isolated per backend."""
    repo_dir = Path(repo_dir)
    build_dir = repo_dir / "build"
    try:
        if not repo_dir.exists():
            cmd = ["git", "clone", "--depth", "1"]
            if branch:
                cmd += ["--branch", branch]
            cmd += [repo_url, str(repo_dir)]
            subprocess.check_call(cmd)
            print(f"   ✓ Cloned {label}")

        cli = find_binary(build_dir, "llama-cli") if build_dir.exists() else None
        if cli is None:
            print(f"🔧 Building {label} llama.cpp (CPU-only)...")
            subprocess.check_call([
                "cmake", "-S", str(repo_dir), "-B", str(build_dir),
                "-DGGML_NATIVE=ON",
                "-DGGML_CUDA=OFF",
                "-DLLAMA_CURL=OFF",
                "-DCMAKE_BUILD_TYPE=Release",
                "-DBUILD_SHARED_LIBS=OFF",
            ])
            # llama-cli is mandatory; llama-bench is optional because some forks
            # do not expose the benchmark target under the same name.
            subprocess.check_call([
                "cmake", "--build", str(build_dir),
                "--config", "Release",
                "--target", "llama-cli",
                "-j", str(MAX_BUILD_JOBS),
            ])
            try:
                subprocess.check_call([
                    "cmake", "--build", str(build_dir),
                    "--config", "Release",
                    "--target", "llama-bench",
                    "-j", str(MAX_BUILD_JOBS),
                ])
            except Exception as bench_exc:
                print(f"   ℹ️  {label} llama-bench skipped: {bench_exc}")

        cli = find_binary(build_dir, "llama-cli")
        bench = find_binary(build_dir, "llama-bench")
        if not cli:
            raise FileNotFoundError(f"{label}: llama-cli was not produced")
        print(f"✅ {label} llama-cli: {cli}")
        if bench:
            print(f"✅ {label} llama-bench: {bench}")
        return cli, bench
    except Exception as exc:
        print(f"⚠️  {label} backend unavailable: {exc}")
        return None, None


# PrismML fork: required for Bonsai Q1_0.
PRISM_LLAMA_CLI, PRISM_LLAMA_BENCH = build_llama_cpp(
    label="PrismML",
    repo_url="https://github.com/PrismML-Eng/llama.cpp.git",
    repo_dir=WORK_DIR / "prism-llama-cpp",
)

# Nanbeige fork: required for the Nanbeige looped-transformer GGUF architecture.
NANBEIGE_LLAMA_CLI, NANBEIGE_LLAMA_BENCH = build_llama_cpp(
    label="Nanbeige42",
    repo_url="https://github.com/Nanbeige/llama.cpp.git",
    repo_dir=WORK_DIR / "nanbeige-llama-cpp",
    branch="nanbeige42",
)

print("\nBackend status:")
print(f"  PrismML CLI  : {PRISM_LLAMA_CLI or 'NOT AVAILABLE'}")
print(f"  Nanbeige CLI : {NANBEIGE_LLAMA_CLI or 'NOT AVAILABLE'}")


In [ ]:
import os, gc, re, time, json, warnings, subprocess, threading, tempfile, ctypes
import psutil, numpy as np, pandas as pd
from pathlib import Path
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

warnings.filterwarnings("ignore")
MODEL_DIR = Path("/kaggle/working/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Cap threads to avoid oversubscription on shared CPU runners.
N_THREADS = max(1, min(8, os.cpu_count() or 4))
RAM_BUDGET_MB = 8 * 1024
RAM_RESERVE_MB = 900
print(f"CPU threads: {N_THREADS}  |  RAM budget: {RAM_BUDGET_MB} MB")


def resolve_binary(root, name="llama-cli"):
    root = Path(root)
    for candidate in [root / "build" / "bin" / name, root / "build" / name]:
        if candidate.is_file():
            return str(candidate)
    build = root / "build"
    if build.exists():
        for candidate in build.rglob(name):
            if candidate.is_file():
                return str(candidate)
    return None


# Re-resolve binaries so this cell also works after a notebook restart.
PRISM_LLAMA_CLI = globals().get("PRISM_LLAMA_CLI") or resolve_binary("/kaggle/working/prism-llama-cpp")
NANBEIGE_LLAMA_CLI = globals().get("NANBEIGE_LLAMA_CLI") or resolve_binary("/kaggle/working/nanbeige-llama-cpp")
CLI_BINARIES = {
    "prism": PRISM_LLAMA_CLI,
    "nanbeige": NANBEIGE_LLAMA_CLI,
}
print(f"PrismML llama-cli : {PRISM_LLAMA_CLI or 'NOT FOUND'}")
print(f"Nanbeige llama-cli: {NANBEIGE_LLAMA_CLI or 'NOT FOUND'}")

MODELS = {
    "Qwen3-4B-Thinking": {
        "repo": "Qwen/Qwen3-4B-GGUF",
        "file": "Qwen3-4B-Q4_K_M.gguf",
        "source_model": "Qwen/Qwen3-4B",
        "template": "qwen3",
        "stop": ["<|im_end|>"],
        "thinking": True,
        "color": "#E87D2E",
        "backend": "python",
        "runtime": "llama-cpp-python",
        "quant": "Q4_K_M",
        "params": "4B",
    },
    "Phi-4-Mini-Reasoning": {
        "repo": "lmstudio-community/Phi-4-mini-reasoning-GGUF",
        "file": "Phi-4-mini-reasoning-Q4_K_M.gguf",
        "source_model": "microsoft/Phi-4-mini-reasoning",
        "template": "phi4",
        "stop": ["<|end|>", "<|endoftext|>"],
        "thinking": False,
        "color": "#0078D4",
        "backend": "python",
        "runtime": "llama-cpp-python",
        "quant": "Q4_K_M",
        "params": "3.8B",
    },
    "Gemma-3-4B-IT": {
        "repo": "lmstudio-community/gemma-3-4b-it-GGUF",
        "file": "gemma-3-4b-it-Q4_K_M.gguf",
        "source_model": "google/gemma-3-4b-it",
        "template": "gemma",
        "stop": ["<end_of_turn>"],
        "thinking": False,
        "color": "#34A853",
        "backend": "python",
        "runtime": "llama-cpp-python",
        "quant": "Q4_K_M",
        "params": "4B",
    },
    "Nanbeige4.2-3B": {
        # Official source model requested by the user; Q4 quant keeps it within 8 GB RAM.
        "repo": "Abiray/Nanbeige4.2-3B-GGUF",
        "file": "Nanbeige4.2-3B-Q4_K_M.gguf",
        "source_model": "Nanbeige/Nanbeige4.2-3B",
        "template": "nanbeige",
        "stop": ["<|im_end|>", "<|endoftext|>"],
        "thinking": True,
        "color": "#C4479D",
        "backend": "cli",
        "engine": "nanbeige",
        "runtime": "Nanbeige llama.cpp CLI",
        "quant": "Q4_K_M",
        "params": "3B non-embedding / 4B total",
    },
    "Bonsai-27B": {
        "repo": "prism-ml/Bonsai-27B-gguf",
        "file": "Bonsai-27B-Q1_0.gguf",
        "source_model": "prism-ml/Bonsai-27B",
        "template": "qwen3",
        "stop": ["<|im_end|>"],
        "thinking": True,
        "color": "#9C27B0",
        "backend": "cli",
        "engine": "prism",
        "runtime": "PrismML llama.cpp CLI",
        "quant": "Q1_0 (ternary)",
        "params": "27B",
    },
}

QUESTIONS = [
    {"id": "MATH-01", "domain": "Agriculture Math",
     "q": "A farmer in Kano, Nigeria plants maize on 4 hectares. Each hectare yields 2.8 tonnes. He sells 70% of the harvest at ₦85,000/tonne and stores the rest. Calculate his total revenue and the mass stored."},
    {"id": "SCI-01", "domain": "Climate Science",
     "q": "The Sahel region is experiencing desertification. Describe the positive feedback loop between vegetation loss and reduced rainfall, then propose two evidence-based reforestation strategies used successfully in Africa."},
    {"id": "MATH-02", "domain": "Public Health Math",
     "q": "A malaria study in Ghana found a prevalence rate of 23% among children under 5. The district has 14,800 such children. If treatment costs $4.20 per child, what is the total treatment cost? Express in USD and GHS (rate: 1 USD = 15.2 GHS)."},
    {"id": "SCI-02", "domain": "Soil Science",
     "q": "Explain why nitrogen-fixing legumes such as cowpea are intercropped with maize across sub-Saharan Africa. Include the biochemical mechanism of nitrogen fixation and quantify the typical kg N/ha contribution per season."},
    {"id": "MATH-03", "domain": "Energy Math",
     "q": "A solar micro-grid in rural Kenya has 12 panels each rated at 300 W. Average daily sunshine is 5.5 hours. If a household needs 3.6 kWh/day, how many households can the micro-grid serve? Show your working step by step."},
]

BM_CFG = {
    "max_tokens": 512,
    "temperature": 0.0,
    "top_p": 1.0,
    "top_k": 0,
    "ctx_size": 4096,
    "n_batch": 256,
    "repeat_penalty": 1.1,
    "timeout_s": 900,
    "seed": 42,
}
print("Config loaded ✅")


In [ ]:
def make_prompt(template, question):
    sys_msg = "You are MUFASAR, an African scientific reasoning assistant. Think step by step."
    if template in {"qwen3", "nanbeige"}:
        # Nanbeige4.2 uses the same <|im_start|>/<|im_end|> role framing.
        return (f"<|im_start|>system\n{sys_msg}<|im_end|>\n"
                f"<|im_start|>user\n{question}<|im_end|>\n"
                f"<|im_start|>assistant\n")
    if template == "phi4":
        return (f"<|system|>\n{sys_msg}<|end|>\n"
                f"<|user|>\n{question}<|end|>\n"
                f"<|assistant|>\n")
    if template == "gemma":
        return (f"<start_of_turn>user\n{sys_msg}\n\n{question}<end_of_turn>\n"
                f"<start_of_turn>model\n")
    return question


def count_thinking_tokens(text, llm_or_none=None):
    match = re.search(r"<think>(.*?)</think>", text, re.DOTALL | re.IGNORECASE)
    if not match:
        return 0
    think_text = match.group(1)
    if llm_or_none is not None:
        try:
            return len(llm_or_none.tokenize(think_text.encode("utf-8")))
        except Exception:
            pass
    return len(think_text.split())


def process_tree_rss_mb(root_pid=None):
    """Total RSS of the notebook process and all live children."""
    try:
        root = psutil.Process(root_pid or os.getpid())
        procs = [root] + root.children(recursive=True)
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        return 0.0
    total = 0
    for proc in procs:
        try:
            total += proc.memory_info().rss
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            continue
    return total / 1024**2


class PeakMemorySampler:
    def __init__(self, interval_s=0.03):
        self.interval_s = interval_s
        self.initial_mb = process_tree_rss_mb()
        self.peak_mb = self.initial_mb
        self._stop = threading.Event()
        self._thread = None

    def _run(self):
        while not self._stop.is_set():
            self.peak_mb = max(self.peak_mb, process_tree_rss_mb())
            self._stop.wait(self.interval_s)

    def start(self):
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def stop(self):
        self._stop.set()
        if self._thread:
            self._thread.join(timeout=1.0)
        self.peak_mb = max(self.peak_mb, process_tree_rss_mb())
        return self


CLI_HELP_CACHE = {}


def cli_help(binary):
    if binary not in CLI_HELP_CACHE:
        try:
            result = subprocess.run(
                [binary, "--help"], capture_output=True, text=True, timeout=30
            )
            CLI_HELP_CACHE[binary] = (result.stdout or "") + (result.stderr or "")
        except Exception:
            CLI_HELP_CACHE[binary] = ""
    return CLI_HELP_CACHE[binary]


def add_flag_if_supported(cmd, binary, flag, *values):
    if flag in cli_help(binary):
        cmd += [flag, *map(str, values)]


def run_inference_python(llm, prompt, cfg, stop):
    sampler = PeakMemorySampler().start()
    t0 = time.perf_counter()
    ttft = None
    chunks = []
    try:
        for chunk in llm.create_completion(
            prompt,
            max_tokens=cfg["max_tokens"],
            temperature=cfg["temperature"],
            top_p=cfg["top_p"],
            top_k=cfg["top_k"],
            repeat_penalty=cfg["repeat_penalty"],
            seed=cfg["seed"],
            stop=stop,
            stream=True,
        ):
            token_text = chunk["choices"][0]["text"]
            if token_text and ttft is None:
                ttft = time.perf_counter() - t0
            chunks.append(token_text)
    finally:
        sampler.stop()

    total_s = time.perf_counter() - t0
    full_text = "".join(chunks)
    try:
        n_out = len(llm.tokenize(full_text.encode("utf-8")))
    except Exception:
        n_out = len(full_text.split())
    try:
        n_in = len(llm.tokenize(prompt.encode("utf-8")))
    except Exception:
        n_in = len(prompt.split())
    return {
        "ttft_s": round(ttft or 0.0, 3),
        "total_s": round(total_s, 3),
        "tps": round(n_out / total_s, 2) if total_s else 0.0,
        "prompt_tok": n_in,
        "output_tok": n_out,
        "peak_ram_mb": round(sampler.peak_mb, 1),
        "ram_delta_mb": round(sampler.peak_mb - sampler.initial_mb, 1),
        "text": full_text,
    }


ANSI_RE = re.compile(r"\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])")


def kill_process_tree(pid):
    try:
        parent = psutil.Process(pid)
        children = parent.children(recursive=True)
        for child in children:
            child.kill()
        parent.kill()
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass


def run_inference_cli(binary, model_path, prompt, cfg, n_threads, stop):
    if not binary:
        raise RuntimeError("Required llama-cli backend is unavailable")

    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".txt", dir="/kaggle/working", delete=False, encoding="utf-8"
    ) as handle:
        handle.write(prompt)
        prompt_file = handle.name

    cmd = [
        binary, "-m", model_path, "-f", prompt_file,
        "-n", str(cfg["max_tokens"]),
        "-c", str(cfg["ctx_size"]),
        "-b", str(cfg["n_batch"]),
        "-t", str(n_threads),
        "--temp", str(cfg["temperature"]),
        "--top-p", str(cfg["top_p"]),
        "--top-k", str(cfg["top_k"]),
        "--repeat-penalty", str(cfg["repeat_penalty"]),
        "--no-display-prompt",
        "-ngl", "0",
    ]
    add_flag_if_supported(cmd, binary, "--seed", cfg["seed"])
    add_flag_if_supported(cmd, binary, "--simple-io")
    add_flag_if_supported(cmd, binary, "--no-warmup")

    sampler = PeakMemorySampler().start()
    t0 = time.perf_counter()
    proc = None
    try:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        try:
            stdout_text, stderr_text = proc.communicate(timeout=cfg["timeout_s"])
        except subprocess.TimeoutExpired:
            kill_process_tree(proc.pid)
            stdout_text, stderr_text = proc.communicate()
            raise TimeoutError(f"llama-cli exceeded {cfg['timeout_s']} seconds")
    finally:
        total_s = time.perf_counter() - t0
        sampler.stop()
        try:
            os.unlink(prompt_file)
        except OSError:
            pass

    if proc.returncode != 0:
        tail = "\n".join((stderr_text or "").splitlines()[-20:])
        raise RuntimeError(f"llama-cli exited with code {proc.returncode}:\n{tail}")

    stdout_text = ANSI_RE.sub("", stdout_text or "").strip()
    stderr_text = ANSI_RE.sub("", stderr_text or "")

    # llama.cpp timing lines exclude model-load time and are more comparable to
    # the already-loaded Python backend. TTFT is approximated as prompt eval time.
    ttft = 0.0
    tps = 0.0
    n_prompt = 0
    n_eval = 0
    prompt_patterns = [
        r"prompt eval time\s*=\s*([\d.]+)\s*ms\s*/\s*(\d+)\s*tokens",
        r"prompt eval time\s*=\s*([\d.]+)\s*ms\s*/\s*(\d+)\s*runs",
    ]
    eval_patterns = [
        r"(?<!prompt )eval time\s*=\s*([\d.]+)\s*ms\s*/\s*(\d+)\s*(?:runs|tokens)",
    ]
    for pattern in prompt_patterns:
        match = re.search(pattern, stderr_text)
        if match:
            ttft = float(match.group(1)) / 1000.0
            n_prompt = int(match.group(2))
            break
    for pattern in eval_patterns:
        matches = list(re.finditer(pattern, stderr_text))
        if matches:
            match = matches[-1]
            eval_ms = float(match.group(1))
            n_eval = int(match.group(2))
            if eval_ms > 0:
                tps = n_eval / (eval_ms / 1000.0)
            break

    if n_eval <= 0:
        n_eval = max(0, len(stdout_text.split()))
    if tps <= 0 and total_s > 0:
        tps = n_eval / total_s
    if n_prompt <= 0:
        n_prompt = len(prompt.split())

    return {
        "ttft_s": round(ttft, 3),
        "total_s": round(total_s, 3),
        "tps": round(tps, 2),
        "prompt_tok": n_prompt,
        "output_tok": n_eval,
        "peak_ram_mb": round(sampler.peak_mb, 1),
        "ram_delta_mb": round(sampler.peak_mb - sampler.initial_mb, 1),
        "text": stdout_text,
    }


def release_memory():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass


def benchmark_model(name, cfg_model):
    repo, filename = cfg_model["repo"], cfg_model["file"]
    backend = cfg_model.get("backend", "python")
    model_path = MODEL_DIR / filename

    if not model_path.exists():
        print(f"Downloading {name} ({repo}/{filename}) …")
        downloaded = hf_hub_download(
            repo_id=repo,
            filename=filename,
            local_dir=str(MODEL_DIR),
        )
        model_path = Path(downloaded)

    size_mb = model_path.stat().st_size / 1024**2
    print(f"\n{'='*68}")
    print(f" {name} | {size_mb/1024:.2f} GiB | {cfg_model['quant']} | {backend}")
    print(f"{'='*68}")

    # Conservative preflight: model file + runtime/KV overhead + notebook reserve.
    estimated_need_mb = size_mb + 1100
    free_mb = psutil.virtual_memory().available / 1024**2
    if estimated_need_mb + RAM_RESERVE_MB > RAM_BUDGET_MB:
        raise MemoryError(
            f"Estimated requirement {estimated_need_mb + RAM_RESERVE_MB:.0f} MB "
            f"exceeds the configured 8 GB budget"
        )
    if free_mb < estimated_need_mb:
        raise MemoryError(
            f"Only {free_mb:.0f} MB is currently available; about "
            f"{estimated_need_mb:.0f} MB is required"
        )

    results = []
    if backend == "cli":
        engine = cfg_model.get("engine")
        binary = CLI_BINARIES.get(engine)
        if not binary:
            raise RuntimeError(f"{engine} llama-cli backend is unavailable")

        for question in QUESTIONS:
            prompt = make_prompt(cfg_model["template"], question["q"])
            print(f"  [{question['id']}] {question['domain']} … ", end="", flush=True)
            result = run_inference_cli(
                binary=binary,
                model_path=str(model_path),
                prompt=prompt,
                cfg=BM_CFG,
                n_threads=N_THREADS,
                stop=cfg_model["stop"],
            )
            result["think_tok"] = (
                count_thinking_tokens(result["text"]) if cfg_model["thinking"] else 0
            )
            result.update(model=name, q_id=question["id"], domain=question["domain"])
            print(
                f"TTFT {result['ttft_s']}s | {result['tps']} tok/s | "
                f"peak total RSS {result['peak_ram_mb']} MB"
            )
            results.append(result)
        release_memory()
        return results

    llm = None
    try:
        llm = Llama(
            model_path=str(model_path),
            n_ctx=BM_CFG["ctx_size"],
            n_batch=BM_CFG["n_batch"],
            n_threads=N_THREADS,
            n_threads_batch=N_THREADS,
            n_gpu_layers=0,
            use_mmap=True,
            use_mlock=False,
            seed=BM_CFG["seed"],
            verbose=False,
        )
        _ = llm("Hello", max_tokens=4, echo=False, temperature=0.0)
        for question in QUESTIONS:
            prompt = make_prompt(cfg_model["template"], question["q"])
            print(f"  [{question['id']}] {question['domain']} … ", end="", flush=True)
            result = run_inference_python(llm, prompt, BM_CFG, cfg_model["stop"])
            result["think_tok"] = (
                count_thinking_tokens(result["text"], llm) if cfg_model["thinking"] else 0
            )
            result.update(model=name, q_id=question["id"], domain=question["domain"])
            print(
                f"TTFT {result['ttft_s']}s | {result['tps']} tok/s | "
                f"peak total RSS {result['peak_ram_mb']} MB"
            )
            results.append(result)
    finally:
        if llm is not None:
            del llm
        release_memory()
        print(f"  Unloaded. Free RAM: {psutil.virtual_memory().available/1024**2:.0f} MB\n")
    return results


print("Helper functions ready ✅")


In [ ]:
ALL_RESULTS = []
MODEL_ERRORS = []

for model_name, model_cfg in MODELS.items():
    try:
        model_results = benchmark_model(model_name, model_cfg)
        ALL_RESULTS.extend(model_results)
    except Exception as exc:
        MODEL_ERRORS.append({"model": model_name, "error": repr(exc)})
        print(f"❌ {model_name} failed: {exc}")
        release_memory()

if not ALL_RESULTS:
    raise RuntimeError(f"No model completed successfully. Errors: {MODEL_ERRORS}")

df = pd.DataFrame(ALL_RESULTS)
df.to_csv("/kaggle/working/mufasa_adtc_benchmark_raw.csv", index=False)

if MODEL_ERRORS:
    pd.DataFrame(MODEL_ERRORS).to_csv(
        "/kaggle/working/mufasa_adtc_model_errors.csv", index=False
    )
    print("⚠️  Some models were skipped; see mufasa_adtc_model_errors.csv")

print("Raw results saved ✅")
df.head()


In [ ]:
from tabulate import tabulate

summary = (
    df.groupby("model", as_index=False)
    .agg(
        TTFT_mean=("ttft_s", "mean"),
        TTFT_std=("ttft_s", "std"),
        TPS_mean=("tps", "mean"),
        TPS_std=("tps", "std"),
        TotalTime=("total_s", "mean"),
        PeakRAM_MB=("peak_ram_mb", "max"),
        RAMDelta_MB=("ram_delta_mb", "max"),
        OutTokens=("output_tok", "mean"),
        ThinkTok=("think_tok", "mean"),
    )
    .round(2)
)
summary.to_csv("/kaggle/working/mufasa_adtc_benchmark_summary.csv", index=False)

print("\n" + "="*78)
print("  MUFASA — Multi-Model Benchmark Summary (8 GB RAM / CPU-only)")
print("="*78)
print(tabulate(summary, headers="keys", tablefmt="rounded_outline", showindex=False))

if MODEL_ERRORS:
    print("\nModels not included:")
    print(tabulate(pd.DataFrame(MODEL_ERRORS), headers="keys", tablefmt="rounded_outline", showindex=False))


In [ ]:
import json
from pathlib import Path

SUBMISSION_DIR = Path("/kaggle/working/mufasa-submission")
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

successful_models = set(summary["model"])

# Build per-model ADTC reports.
for model_name in summary["model"]:
    row = summary.loc[summary["model"] == model_name].iloc[0]
    cfg = MODELS[model_name]
    report = {
        "team_id": "mufasa-team",
        "domain": "math_scientific_reasoning",
        "language_scope": ["en"],
        "budget_laptop_claim": True,
        "model": {
            "name": model_name,
            "source_model": cfg.get("source_model", cfg["repo"]),
            "artifact_repo": cfg["repo"],
            "runtime": cfg["runtime"],
            "quantization": cfg["quant"],
            "parameters_estimate": cfg["params"],
        },
        "throughput": {
            "tokens_per_second_generation": float(row["TPS_mean"]),
            "first_token_latency_ms": round(float(row["TTFT_mean"]) * 1000, 1),
        },
        "memory": {
            "peak_total_notebook_rss_mb": float(row["PeakRAM_MB"]),
            "max_inference_rss_delta_mb": float(row["RAMDelta_MB"]),
        },
        "constraint": "CPU-only, 8 GB RAM",
    }
    slug = re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_")
    out_path = SUBMISSION_DIR / f"{slug}_submission.json"
    with out_path.open("w", encoding="utf-8") as handle:
        json.dump(report, handle, indent=2, ensure_ascii=False)
    print(f"✅ {out_path}")

combined = {
    "team_id": "mufasa-team",
    "constraint": "CPU-only, 8 GB RAM",
    "models": {},
    "errors": MODEL_ERRORS,
}
for model_name in summary["model"]:
    row = summary.loc[summary["model"] == model_name].iloc[0]
    cfg = MODELS[model_name]
    combined["models"][model_name] = {
        "source_model": cfg.get("source_model", cfg["repo"]),
        "quantization": cfg["quant"],
        "runtime": cfg["runtime"],
        "tps": float(row["TPS_mean"]),
        "ttft_ms": round(float(row["TTFT_mean"]) * 1000, 1),
        "peak_total_notebook_rss_mb": float(row["PeakRAM_MB"]),
    }

combined_path = Path("/kaggle/working/mufasa_combined_submission.json")
with combined_path.open("w", encoding="utf-8") as handle:
    json.dump(combined, handle, indent=2, ensure_ascii=False)
print(f"\n✅ Combined report: {combined_path}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle(
    "MUFASA Multi-Model ADTC Benchmark\nAfrica Deep Tech Challenge 2026 — CPU-only / 8 GB RAM",
    fontsize=14, fontweight="bold", y=1.01,
)

colors = [MODELS[model]["color"] for model in summary["model"]]
metrics = [
    ("TPS_mean", "Throughput (tok/s)", "Higher is better ▲", axes[0, 0]),
    ("TTFT_mean", "Time to First Token (s)", "Lower is better ▼", axes[0, 1]),
    ("TotalTime", "Avg Elapsed Time (s)", "Lower is better ▼", axes[0, 2]),
    ("PeakRAM_MB", "Peak Total RSS (MB)", "Lower is better ▼", axes[1, 0]),
    ("OutTokens", "Avg Output Tokens", "Higher = richer", axes[1, 1]),
    ("ThinkTok", "Reasoning Trace Tokens", "Thinking depth", axes[1, 2]),
]

for column, title, note, axis in metrics:
    bars = axis.bar(summary["model"], summary[column], color=colors, edgecolor="white", linewidth=1.2)
    axis.set_title(title, fontweight="bold", fontsize=11)
    axis.tick_params(axis="x", rotation=25, labelsize=8)
    axis.set_ylabel(note, fontsize=8, color="grey")
    for bar, value in zip(bars, summary[column]):
        offset = max(float(bar.get_height()) * 0.02, 0.02)
        axis.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            f"{value:.1f}",
            ha="center", va="bottom", fontsize=8, fontweight="bold",
        )
    if column == "PeakRAM_MB":
        axis.axhline(8192, color="red", linestyle="--", linewidth=1, label="8 GB limit")
        axis.legend(fontsize=8)

plt.tight_layout()
plt.savefig("/kaggle/working/mufasa_adtc_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved ✅")


In [ ]:
pivot = df.pivot_table(index="domain", columns="model", values="tps", aggfunc="mean").round(2)
print("\nThroughput (tok/s) by Domain:\n")
print(tabulate(pivot, headers="keys", tablefmt="rounded_outline"))

best = summary.loc[summary["TPS_mean"].idxmax(), "model"]
fastest_ttft = summary.loc[summary["TTFT_mean"].idxmin(), "model"]
lowest_ram = summary.loc[summary["PeakRAM_MB"].idxmin(), "model"]

lines = [
    "+--------------------------------------------------------------+",
    "|          MUFASA MODEL SELECTION RECOMMENDATION             |",
    "|          (CPU-only / 8 GB RAM constraint)                  |",
    "+--------------------------------------------------------------+",
    f"|  Best throughput : {best:<39}|",
    f"|  Fastest TTFT    : {fastest_ttft:<39}|",
    f"|  Lowest RAM      : {lowest_ram:<39}|",
    "+--------------------------------------------------------------+",
]
print("\n".join(lines))
